**What `02_training.ipynb` does — Big Picture**

**Step 1 — Load the dataset** from disk (same as before)

**Step 2 — Preprocess the images** using `ViTImageProcessor` which:
- Resizes 64×64 → 224×224
- Normalizes pixel values using ImageNet mean/std
- Converts images to tensors

**Step 3 — Load the pre-trained ViT model** from HuggingFace and replace the final classification head with 10 outputs (one per class)

**Step 4 — Define metrics** so we can track accuracy during training

**Step 5 — Configure training arguments** (learning rate, epochs, batch size etc.)

**Step 6 — Train using HuggingFace `Trainer`** which handles the training loop automatically

**Step 7 — Save the model** to disk for use in future notebooks

In [2]:
# Core libraries
from datasets import load_from_disk
import numpy as np

# HuggingFace — model, processor, and training utilities
from transformers import (
    ViTImageProcessor,
    ViTForImageClassification,
    TrainingArguments,
    Trainer
)

# Metrics
from sklearn.metrics import accuracy_score, f1_score

import torch

# Confirm GPU is available
print(f"PyTorch version: {torch.__version__}")
print(f"GPU available: {torch.cuda.is_available()}")
print(f"Device: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")

PyTorch version: 2.8.0+cu128
GPU available: True
Device: NVIDIA L4


In [3]:
# Load dataset from disk
dataset = load_from_disk("../data/eurosat")

# Extract class names
class_names = dataset["train"].features["label"].names
num_classes = len(class_names)

# Create two dictionaries the model needs:
# id2label: given a number, what class is it?  e.g. 0 → "AnnualCrop"
# label2id: given a class name, what number is it? e.g. "AnnualCrop" → 0
id2label = {i: class_names[i] for i in range(num_classes)}
label2id = {class_names[i]: i for i in range(num_classes)}

print(f"Number of classes: {num_classes}")
print(f"\nid2label: {id2label}")
print(f"\nlabel2id: {label2id}")

Number of classes: 10

id2label: {0: 'Annual Crop', 1: 'Forest', 2: 'Herbaceous Vegetation', 3: 'Highway', 4: 'Industrial Buildings', 5: 'Pasture', 6: 'Permanent Crop', 7: 'Residential Buildings', 8: 'River', 9: 'SeaLake'}

label2id: {'Annual Crop': 0, 'Forest': 1, 'Herbaceous Vegetation': 2, 'Highway': 3, 'Industrial Buildings': 4, 'Pasture': 5, 'Permanent Crop': 6, 'Residential Buildings': 7, 'River': 8, 'SeaLake': 9}


In [ ]:
# Load the ViT processor — this handles resizing and normalization automatically
processor = ViTImageProcessor.from_pretrained("google/vit-base-patch16-224")

# Define the preprocessing function
def preprocess(batch):
    # Convert images to RGB (safety check — ensures no grayscale sneaks in)
    images = [img.convert("RGB") for img in batch["image"]]
    
    # Process images — resizes to 224x224 and normalizes pixel values
    inputs = processor(images=images, return_tensors="pt")
    
    # Add labels to the inputs
    inputs["labels"] = batch["label"]
    
    return inputs

# Apply preprocessing to all splits
# batched=True means it processes multiple images at once — much faster
dataset = dataset.map(preprocess, batched=True, remove_columns=["filename"])

# Tell the dataset to return PyTorch tensors
dataset.set_format("torch", columns=["pixel_values", "labels"])

print("Preprocessing complete!")
print(f"Features: {dataset['train'].features}")

Preprocessing complete!
Features: {'image': Image(mode=None, decode=True), 'label': ClassLabel(names=['Annual Crop', 'Forest', 'Herbaceous Vegetation', 'Highway', 'Industrial Buildings', 'Pasture', 'Permanent Crop', 'Residential Buildings', 'River', 'SeaLake']), 'pixel_values': List(List(List(Value('float32')))), 'labels': Value('int64')}


In [ ]:
# Load the pre-trained ViT model and replace the classification head
# ignore_mismatched_sizes=True — because the original model has 1000 output classes
# (ImageNet) but we need only 10, so we replace the head with a new one
model = ViTForImageClassification.from_pretrained(
    "google/vit-base-patch16-224",
    num_labels=num_classes,
    id2label=id2label,
    label2id=label2id,
    ignore_mismatched_sizes=True
)

# Print model summary
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Total parameters:     {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")

Loading weights:   0%|          | 0/200 [00:00<?, ?it/s]

ViTForImageClassification LOAD REPORT from: google/vit-base-patch16-224
Key               | Status   |                                                                                           
------------------+----------+-------------------------------------------------------------------------------------------
classifier.weight | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([1000, 768]) vs model:torch.Size([10, 768])
classifier.bias   | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([1000]) vs model:torch.Size([10])          

Notes:
- MISMATCH	:ckpt weights were loaded, but they did not match the original empty weight shapes.


Total parameters:     85,806,346
Trainable parameters: 85,806,346


In [6]:
# Define the function that computes metrics during evaluation
# The Trainer calls this automatically after each epoch
def compute_metrics(eval_pred):
    # eval_pred contains two things:
    # logits — raw scores output by the model (one score per class)
    # labels — the true class numbers
    logits, labels = eval_pred
    
    # Convert raw scores to predicted class numbers
    # argmax picks the index of the highest score
    # e.g. [0.1, 0.8, 0.05, ...] → 1 (Forest)
    predictions = np.argmax(logits, axis=-1)
    
    # Accuracy — what % of predictions are correct
    accuracy = accuracy_score(labels, predictions)
    
    # F1 Score — balances precision and recall per class
    # macro = calculates F1 for each class separately then averages
    # important because our dataset has slight class imbalance (remember Pasture!)
    f1 = f1_score(labels, predictions, average="macro")
    
    return {
        "accuracy": round(accuracy, 4),
        "f1": round(f1, 4)
    }

print("Metrics function defined!")
print("Tracking: Accuracy and F1 Score")

Metrics function defined!
Tracking: Accuracy and F1 Score


In [ ]:
training_args = TrainingArguments(
    # Folder where model checkpoints will be saved during training
    output_dir="../models/satellite-vit",
    
    # Number of times the model sees the entire training set
    num_train_epochs=10,
    
    # Number of images processed at once — 32 is safe for most GPUs
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    
    # How often to evaluate — after every epoch
    eval_strategy="epoch",         # renamed from evaluation_strategy
    
    # Save a checkpoint after every epoch
    save_strategy="epoch",
    
    # Learning rate — how big each weight update step is
    learning_rate=2e-5,
    
    # Load the best model at the end (not necessarily the last epoch)
    load_best_model_at_end=True,
    
    # Use accuracy to decide which checkpoint is "best"
    metric_for_best_model="accuracy",
    
    # Only keep the 2 best checkpoints to save disk space
    save_total_limit=2,
    
    # Print training logs every 100 steps
    logging_steps=100,
    
    # Remove unused columns from dataset automatically
    remove_unused_columns=False,
    
    # Seed for reproducibility — same results every run
    seed=42,
)

print("Training arguments configured!")
print(f"  Epochs:        {training_args.num_train_epochs}")
print(f"  Batch size:    {training_args.per_device_train_batch_size}")
print(f"  Learning rate: {training_args.learning_rate}")
print(f"  Output dir:    {training_args.output_dir}")

Training arguments configured!
  Epochs:        10
  Batch size:    32
  Learning rate: 2e-05
  Output dir:    ../models/satellite-vit


In [8]:
# Initialize the Trainer — brings everything together
trainer = Trainer(
    model=model,                                  # our ViT model
    args=training_args,                           # training configuration
    train_dataset=dataset["train"],               # training data
    eval_dataset=dataset["validation"],           # validation data
    compute_metrics=compute_metrics,              # accuracy and F1
)

# Start training!
# This is where the magic happens 🚀
trainer.train()


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.073517,0.072780,0.974300,0.972800
2,0.040534,0.048480,0.985700,0.985100
3,0.016168,0.051418,0.985700,0.985000
4,0.004455,0.061584,0.985900,0.985300
5,0.000633,0.056458,0.988500,0.988000
6,0.000253,0.051565,0.988300,0.987800
7,0.000185,0.051152,0.989100,0.988500
8,0.000142,0.051600,0.989300,0.988800
9,0.000120,0.052044,0.989300,0.988800
10,0.000120,0.052275,0.989300,0.988800


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=5070, training_loss=0.02915540738435392, metrics={'train_runtime': 4898.3802, 'train_samples_per_second': 33.072, 'train_steps_per_second': 1.035, 'total_flos': 1.2554602436468736e+19, 'train_loss': 0.02915540738435392, 'epoch': 10.0})

"Training loss kept decreasing but validation loss started rising after epoch 2 — a classic sign of overfitting. Because I set load_best_model_at_end=True, the trainer automatically kept the best checkpoint rather than the final one."

In [3]:
import os

# List everything inside the models folder to find the checkpoint
for root, dirs, files in os.walk("../models"):
    for f in files:
        print(os.path.join(root, f))

../models/satellite-vit/checkpoint-5070/training_args.bin
../models/satellite-vit/checkpoint-5070/model.safetensors
../models/satellite-vit/checkpoint-5070/trainer_state.json
../models/satellite-vit/checkpoint-5070/optimizer.pt
../models/satellite-vit/checkpoint-5070/rng_state.pth
../models/satellite-vit/checkpoint-5070/config.json
../models/satellite-vit/checkpoint-5070/scheduler.pt
../models/satellite-vit/checkpoint-4056/training_args.bin
../models/satellite-vit/checkpoint-4056/model.safetensors
../models/satellite-vit/checkpoint-4056/trainer_state.json
../models/satellite-vit/checkpoint-4056/optimizer.pt
../models/satellite-vit/checkpoint-4056/rng_state.pth
../models/satellite-vit/checkpoint-4056/config.json
../models/satellite-vit/checkpoint-4056/scheduler.pt


In [4]:
from transformers import ViTForImageClassification, ViTImageProcessor

# Load from the best checkpoint (epoch 8, not the last one)
checkpoint_path = "../models/satellite-vit/checkpoint-4056"

model = ViTForImageClassification.from_pretrained(checkpoint_path)
processor = ViTImageProcessor.from_pretrained("google/vit-base-patch16-224")

print("✅ Best model reloaded from checkpoint-4056 (Epoch 8)!")

Loading weights:   0%|          | 0/200 [00:00<?, ?it/s]

✅ Best model reloaded from checkpoint-4056 (Epoch 8)!


In [5]:
import os

os.makedirs("../models/satellite-vit-final", exist_ok=True)
model.save_pretrained("../models/satellite-vit-final")
processor.save_pretrained("../models/satellite-vit-final")

print("✅ Model saved to ../models/satellite-vit-final")
print("Contents:")
for f in os.listdir("../models/satellite-vit-final"):
    print(f"  {f}")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

✅ Model saved to ../models/satellite-vit-final
Contents:
  model.safetensors
  preprocessor_config.json
  config.json
